# Algoritmo Exato
---

In [6]:
import sys
from pathlib import Path

diretorio_atual = Path.cwd()

pasta_src = diretorio_atual / "article" / "src"

if str(pasta_src) not in sys.path:
    sys.path.append(str(pasta_src))

from metamodel import MetaModel

In [7]:
import pandas as pd

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, f1_score

In [8]:
meta_dataset = pd.read_csv('data/metafeatures_dataset_with_best.csv', index_col=0)

In [ ]:
metamodels = {
    "decision_tree": DecisionTreeClassifier(random_state=42),
    "random_forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "hist_gradient": HistGradientBoostingClassifier(random_state=42),
    "svm": make_pipeline(StandardScaler(), SVC(kernel='rbf', random_state=42)),
    "knn": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=3))
}

In [10]:
for key, value in metamodels.items():
    
    metamodel = MetaModel()
    print(f"### {key} ###")
    
    predicts = metamodel.train_and_evaluate_best_metamodel(meta_dataset, model=value)
    
    y_pred = list(predicts['Best clf (pred)'])
    y_true = meta_dataset['Best'].values
    
    metamodel_accuracy = accuracy_score(y_true, y_pred)
    metamodel_f1 = f1_score(y_true, y_pred, average='weighted')
    
    print(f'Meta-model Accuracy: {metamodel_accuracy:.2f}')
    print(f'Meta-model F1-score: {metamodel_f1:.2f}')
    
    print(f"####{len(key)*'#'}####")

### decision_tree ###
Meta-model Accuracy: 0.43
Meta-model F1-score: 0.43
#####################
### random_forest ###
Meta-model Accuracy: 0.53
Meta-model F1-score: 0.49
#####################
### svm ###
Meta-model Accuracy: 0.54
Meta-model F1-score: 0.48
###########
### hist_gradient ###
Meta-model Accuracy: 0.47
Meta-model F1-score: 0.45
#####################
### knn ###
Meta-model Accuracy: 0.39
Meta-model F1-score: 0.37
###########


# Modelagem Binaria
---

In [ ]:
import numpy as np
import pandas as pd
from pulp import *
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
import warnings

warnings.filterwarnings("ignore")

def model_mat(max_feature: int = 10, l: float = 0.01):
    
    X = meta_dataset[meta_feature_cols]
    y = meta_dataset['Best']
    
    rf = RandomForestClassifier(n_estimators=200, random_state=42)
    rf.fit(X, y)
    feature_scores = rf.feature_importances_
    problem = LpProblem("MetaFeature_Selection", LpMaximize)

    x = {i: LpVariable(f"{meta_feature_cols[i]}", cat="Binary") for i in range(len(meta_feature_cols))}

    # Hiperparâmetros
    MAX_FEATURES = max_feature  # Número máximo de meta-features que você deseja
    LAMBDA = l     # Penalidade para features com importância muito baixa

    problem += train_and_evaluate_meta_model(selected_features) + lpSum((feature_scores[i] - LAMBDA) * x[i] for i in range(len(meta_feature_cols)))

    problem += lpSum(x[i] for i in range(len(meta_feature_cols))) <= MAX_FEATURES

    problem.solve(PULP_CBC_CMD(msg=False))
    
    selected_features = [meta_feature_cols[i] for i in range(len(meta_feature_cols)) if value(x[i]) == 1]
    

    return selected_features


In [64]:
import numpy as np
import pandas as pd
from pulp import *
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings("ignore")

# =====================================================================
# 1. MODELO MATEMÁTICO HÍBRIDO (Filtro Rápido + PuLP)
# =====================================================================
def model_mat_hybrid(X_train, y_train, max_feature: int, l: float = 0.002, filter_top_k: int = 50):
    """Filtra as top_k características e otimiza a escolha final via PuLP."""
    # Fase 1: Filtro rápido com árvore
    rf_selector = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_selector.fit(X_train, y_train)
    
    all_importances = rf_selector.feature_importances_
    k = min(filter_top_k, len(all_importances))
    top_k_indices = np.argsort(all_importances)[-k:]
    
    top_k_cols = [X_train.columns[i] for i in top_k_indices]
    top_k_scores = all_importances[top_k_indices]

    # Fase 2: Otimização matemática
    problem = LpProblem("MetaFeature_Hybrid_Selection", LpMaximize)
    x = {i: LpVariable(f"feat_{i}", cat="Binary") for i in range(len(top_k_cols))}

    problem += lpSum((top_k_scores[i] - l) * x[i] for i in range(len(top_k_cols)))
    problem += lpSum(x[i] for i in range(len(top_k_cols))) <= max_feature

    problem.solve(PULP_CBC_CMD(msg=False))

    # A função value() do PuLP agora funcionará normalmente
    selected_features = [top_k_cols[i] for i in range(len(top_k_cols)) if value(x[i]) == 1]
    
    if not selected_features:
        selected_features = top_k_cols[:max_feature]
        
    return selected_features


# =====================================================================
# 2. FUNÇÃO DE AVALIAÇÃO DO META-MODELO
# =====================================================================
def train_and_evaluate_meta_model_teste(meta_dataset, model, max_feature: int):
    max_f = 0
    loo = LeaveOneOut()
    y_true = meta_dataset['Best'].values
    y_pred = []

    for train_index, test_index in loo.split(meta_dataset):
        # Separação das colunas de forma dinâmica
        X = meta_dataset.drop(columns=['Dataset', 'Best'] + classifier_cols)
        y = meta_dataset['Best']
        
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]
        
        X_train = X_train.fillna(0)
        X_test = X_test.fillna(0)
        
        # Executa a seleção híbrida baseada no 'max_feature' do loop atual
        features_otimizadas = model_mat_hybrid(
            X_train, y_train, 
            max_feature=max_feature, 
            l=0.002, 
            filter_top_k=50
        )

        if max_f < len(features_otimizadas):
                max_f = len(features_otimizadas)
        
        X_train_sel = X_train[features_otimizadas]
        X_test_sel = X_test[features_otimizadas]
        
        clf = model
        clf.fit(X_train_sel, y_train)
        y_pred.append(clf.predict(X_test_sel)[0])
    
    print("Numero maximo de features: ", max_f)
    meta_model_accuracy = accuracy_score(y_true, y_pred)
    meta_model_f1 = f1_score(y_true, y_pred, average='weighted')
    
    return None, meta_model_accuracy, meta_model_f1

# =====================================================================
# 3. EXECUÇÃO DO LOOP (CORRIGIDO)
# =====================================================================
for i in range(10, len(meta_feature_cols) + 1, 100):
    # Mudamos o nome da variável de 'value' para 'model_instance' para não quebrar o PuLP
    for key, model_instance in meta_models.items(): 
        df_resultado, acuracia, f1 = train_and_evaluate_meta_model_teste(
            meta_dataset=meta_dataset, 
            model=model_instance, 
            max_feature=i
        )
        print("\n" + "="*50)
        print(f"MÉTRICAS: MODELO [{key.upper()}] | MAX FEATURES: {i}")
        print("="*50)
        print(f"Acurácia Geral: {acuracia:.4f}")
        print(f"F1-Score Geral: {f1:.4f}")

Numero maximo de features:  10

MÉTRICAS: MODELO [DECISION_TREE] | MAX FEATURES: 10
Acurácia Geral: 0.4149
F1-Score Geral: 0.4130
Numero maximo de features:  10

MÉTRICAS: MODELO [RANDOM_FOREST] | MAX FEATURES: 10
Acurácia Geral: 0.3723
F1-Score Geral: 0.3545
Numero maximo de features:  10

MÉTRICAS: MODELO [SVM] | MAX FEATURES: 10
Acurácia Geral: 0.4149
F1-Score Geral: 0.3496
Numero maximo de features:  10

MÉTRICAS: MODELO [KNN] | MAX FEATURES: 10
Acurácia Geral: 0.3723
F1-Score Geral: 0.3632
Numero maximo de features:  50

MÉTRICAS: MODELO [DECISION_TREE] | MAX FEATURES: 110
Acurácia Geral: 0.3723
F1-Score Geral: 0.3731
Numero maximo de features:  50

MÉTRICAS: MODELO [RANDOM_FOREST] | MAX FEATURES: 110
Acurácia Geral: 0.4255
F1-Score Geral: 0.4063
Numero maximo de features:  50

MÉTRICAS: MODELO [SVM] | MAX FEATURES: 110
Acurácia Geral: 0.4681
F1-Score Geral: 0.4110


KeyboardInterrupt: 

In [ ]:
from sklearn.model_selection import cross_validate

selected_features_list = []

for max_feature in range(1, len(meta_feature_cols) + 1):
    result = model_mat(max_feature=max_feature, l=0.002)
    
    if result:
        X = meta_dataset[meta_feature_cols].fillna(0)
        y = meta_dataset['Best']
        X_selected = X[result]
        
        clf = RandomForestClassifier(random_state=42)
        
        # Definimos as métricas que queremos
        metrics = ['f1_weighted', 'accuracy']
        
        # Executa a validação cruzada para ambas as métricas
        cv_results = cross_validate(clf, X_selected, y, cv=5, scoring=metrics)
        
        f1_mean = cv_results['test_f1_weighted'].mean()
        acc_mean = cv_results['test_accuracy'].mean()
        
        # Guardamos em um dicionário para ficar fácil de ler e ordenar
        selected_features_list.append({
            'f1_weighted': f1_mean,
            'accuracy': acc_mean,
            'features': result,
            'n_features': len(result)
        })
        
        print(f"Features: {len(result)} | F1: {f1_mean:.4f} | Acc: {acc_mean:.4f}")

Features: 1 | F1: 0.1973 | Acc: 0.2023
Features: 2 | F1: 0.4329 | Acc: 0.4789
Features: 3 | F1: 0.4817 | Acc: 0.5199
Features: 4 | F1: 0.5008 | Acc: 0.5316
Features: 5 | F1: 0.5292 | Acc: 0.5532
Features: 6 | F1: 0.5371 | Acc: 0.5643
Features: 7 | F1: 0.4888 | Acc: 0.5322
Features: 8 | F1: 0.4813 | Acc: 0.5222
Features: 9 | F1: 0.5214 | Acc: 0.5532
Features: 10 | F1: 0.4862 | Acc: 0.5211
Features: 11 | F1: 0.5122 | Acc: 0.5433
Features: 12 | F1: 0.5281 | Acc: 0.5526
Features: 13 | F1: 0.4906 | Acc: 0.5216
Features: 14 | F1: 0.4743 | Acc: 0.5111
Features: 15 | F1: 0.5149 | Acc: 0.5538
Features: 16 | F1: 0.5069 | Acc: 0.5433
Features: 17 | F1: 0.5289 | Acc: 0.5538
Features: 18 | F1: 0.5420 | Acc: 0.5637
Features: 19 | F1: 0.5018 | Acc: 0.5322
Features: 20 | F1: 0.5345 | Acc: 0.5737
Features: 21 | F1: 0.5060 | Acc: 0.5322
Features: 22 | F1: 0.5468 | Acc: 0.5637
Features: 23 | F1: 0.4983 | Acc: 0.5322
Features: 24 | F1: 0.4905 | Acc: 0.5111
Features: 25 | F1: 0.5184 | Acc: 0.5532
Features:

In [ ]:
import json

# Defina o nome do arquivo
nome_arquivo = "data/result_meta_features_choice.json"

# Salvar a lista no arquivo
with open(nome_arquivo, "w", encoding="utf-8") as f:
    json.dump(selected_features_list, f, indent=4, ensure_ascii=False)

print(f"Resultado salvo com sucesso em: {nome_arquivo}")

Resultado salvo com sucesso em: data/result_meta_features_choice.json


In [ ]:
# ==========================================
# ORDENAÇÃO E EXIBIÇÃO
# ==========================================

# Ordenar por F1-Weighted (Decrescente)
selected_features_list.sort(key=lambda x: x['f1_weighted'], reverse=True)

print("\n" + "="*30)
print("RANKING FINAL")
print("="*30)

for i, item in enumerate(selected_features_list[:5]):  # Top 5
    print(f"{i+1}º Lugar ({item['n_features']} features):")
    print(f"   - Acurácia: {item['accuracy']:.4f}")
    print(f"   - F1-Score: {item['f1_weighted']:.4f}")


RANKING FINAL
1º Lugar (22 features):
   - Acurácia: 0.5637
   - F1-Score: 0.5468
2º Lugar (37 features):
   - Acurácia: 0.5749
   - F1-Score: 0.5443
3º Lugar (18 features):
   - Acurácia: 0.5637
   - F1-Score: 0.5420
4º Lugar (79 features):
   - Acurácia: 0.5637
   - F1-Score: 0.5373
5º Lugar (6 features):
   - Acurácia: 0.5643
   - F1-Score: 0.5371


In [13]:
# Célula: localizar e importar `meta_model.py` dinamicamente e executar o MetaModel
import os
import importlib.util
import glob
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score
from IPython.display import display

# Procura o arquivo meta_model.py em todo o workspace (evita caminhos duplicados)
matches = glob.glob('**/meta_model.py', recursive=True)
if not matches:
    print("Arquivo meta_model.py não encontrado no workspace.")
else:
    file_path = matches[0]
    print(f"Usando: {file_path}")
    spec = importlib.util.spec_from_file_location("metamodel", file_path)
    meta_mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(meta_mod)
    MetaModel = getattr(meta_mod, 'MetaModel')

    # Procura o CSV correspondente (tenta alguns caminhos possíveis)
    csv_matches = glob.glob('**/metafeatures_dataset_with_best.csv', recursive=True)
    if not csv_matches:
        print("Dataset metafeatures_dataset_with_best.csv não encontrado no workspace.")
    else:
        data_path = csv_matches[0]
        print(f"Usando dataset: {data_path}")
        dt = DecisionTreeClassifier()
        meta_dataset = pd.read_csv(data_path, index_col=0)
        metamodel = MetaModel()
        df_predicts = metamodel.train_and_evaluate_best_metamodel(meta_dataset, model=dt)
        y_pred = list(df_predicts['Best clf (pred)'])
        y_true = meta_dataset['Best'].values
        print("acuracia:", accuracy_score(y_true, y_pred))
        print("f1 (weighted):", f1_score(y_true, y_pred, average='weighted'))
        display(df_predicts.head())


Arquivo meta_model.py não encontrado no workspace.


In [16]:
import os

# Retorna o diretório de trabalho atual
diretorio_atual = os.getcwd()
print(diretorio_atual)


c:\Users\Victo\Desktop\Ufal\2026.1\cc-meta-aprendizagem-2026.1\article\src


In [ ]:
import sys

# 1. Define o caminho da pasta onde o arquivo .py está

